# Stage 8B — NLM Chest-Radiography Access Completion

This is an **independent, append-only repair stage**. It does not alter the completed Stage 8 protocol, predictions, results, or final record. It verifies the sealed Stage 8 hash, replaces only the two failed NLM directory-root requests with the official explicit `index.html` endpoints, reuses the frozen TBX11K representation and axis, and adds chest-radiography cross-domain edges.

The same frozen ImageNet ResNet50 V2 representation, linear source probe, grouped source gate, label-free components, fixed threshold, and D/C/O transportability definitions are retained. New target predictions are frozen before outcome evaluation. No target refit, threshold tuning, sign reversal, architecture search, feature selection, or final DDO2 fit is authorised.

NLM images are streamed from the official server and never copied to Google Drive. Only manifests, frozen embeddings, axes, predictions, tables, records, and a small figure are persisted.


In [1]:
#@title 08B-0. Mount Drive, verify frozen Stage 8, and seal the access-completion protocol
from google.colab import drive
drive.mount("/content/drive")

import gc
import hashlib
import io
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd


print("================ STAGE 8B NLM ACCESS-COMPLETION PREFLIGHT ================")

PROJECT_ROOT = Path("/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability")
CODE_ROOT = PROJECT_ROOT / "05_Code" / "Cross_Modal"
STAGE8_ROOT = (
    PROJECT_ROOT / "06_Data_Records" / "Cross_Modal" /
    "Stage8_CrossModality_EdgeLibrary_Expansion_v0.1"
)
STAGE8_FINAL_PATH = STAGE8_ROOT / "06_Results" / "Stage8_CrossModality_Expansion_Complete_v0.1.json"
STAGE8_OUTPUT_MANIFEST_PATH = STAGE8_ROOT / "06_Results" / "Stage8_Output_Integrity_Manifest_v0.1.csv"
STAGE8_TBX_MANIFEST_PATH = STAGE8_ROOT / "01_Acquisition_Manifests" / "TBX11K_Harmonised_Acquisition_Manifest_v0.1.csv"
STAGE8_TBX_PARTITION_PATH = STAGE8_ROOT / "01_Acquisition_Manifests" / "TBX11K_Frozen_Development_Validation_Unit_Manifest_v0.1.csv"
STAGE8_TBX_EMBEDDING_PATH = STAGE8_ROOT / "02_Frozen_Embeddings" / "TBX11K_Frozen_ResNet50V2_L2_Embeddings_v0.1.npz"
STAGE8_TBX_AXIS_PATH = STAGE8_ROOT / "03_Frozen_Source_Axes" / "TBX11K_Frozen_Source_Axis_v0.1.npz"
STAGE8_SOURCE_SUMMARY_PATH = STAGE8_ROOT / "03_Frozen_Source_Axes" / "Stage8_Source_Recoverability_Summary_v0.1.csv"
STAGE8_EDGE_LIBRARY_PATH = STAGE8_ROOT / "05_Unsealed_Discovery" / "Stage8_ThreeModality_Eligible_Edge_Library_v0.1.csv"

STAGE8B_ROOT = (
    PROJECT_ROOT / "06_Data_Records" / "Cross_Modal" /
    "Stage8B_NLM_Chest_Radiography_Access_Completion_v0.1"
)
PROTOCOL_ROOT = STAGE8B_ROOT / "00_Protocol"
ACQUISITION_ROOT = STAGE8B_ROOT / "01_NLM_Acquisition"
EMBEDDING_ROOT = STAGE8B_ROOT / "02_Frozen_Embeddings"
AXIS_ROOT = STAGE8B_ROOT / "03_Frozen_Source_Axes"
FREEZE_ROOT = STAGE8B_ROOT / "04_Prediction_Freeze"
DISCOVERY_ROOT = STAGE8B_ROOT / "05_Unsealed_Chest_Discovery"
RESULT_ROOT = STAGE8B_ROOT / "06_Results"
for directory in [CODE_ROOT, PROTOCOL_ROOT, ACQUISITION_ROOT, EMBEDDING_ROOT, AXIS_ROOT, FREEZE_ROOT, DISCOVERY_ROOT, RESULT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK_PATH = CODE_ROOT / "CrossModal_Stage8B_NLM_Chest_Radiography_Access_Completion_v0.1.ipynb"
PROTOCOL_SEAL_PATH = PROTOCOL_ROOT / "Stage8B_NLM_Access_Completion_Protocol_Seal_v0.1.json"
DATASET_REGISTRY_PATH = PROTOCOL_ROOT / "Stage8B_Frozen_NLM_Dataset_Registry_v0.1.csv"
INPUT_COMMITMENT_PATH = PROTOCOL_ROOT / "Stage8B_Input_Integrity_Commitment_v0.1.csv"
RUNTIME_STATE_PATH = RESULT_ROOT / "Stage8B_Runtime_State_v0.1.json"
FINAL_RECORD_PATH = RESULT_ROOT / "Stage8B_NLM_Access_Completion_Complete_v0.1.json"
TEMP_ROOT = Path("/content/stage8b_tmp")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

EXPECTED_STAGE8_FINAL_HASH = "698c019b8516a84522831216b6cb0c85e047e6d0b401071db99e1d74ae6b2397"
RANDOM_SEED = 20260721
N_BOOTSTRAP = 400
FROZEN_FEATURE_DIMENSION = 2048
FIXED_THRESHOLD = 0.50
AUC_MINIMUM = 0.70
AUC_CI_LOWER_STRICT_MINIMUM = 0.55
CALIBRATION_DEGRADATION_TOLERANCE = 0.05
OPERATING_POINT_BALANCED_ACCURACY_MINIMUM = 0.70
DOWNLOAD_WORKERS = 4
EMBEDDING_BATCH_SIZE = 16
MAXIMUM_NEW_STAGE8B_BYTES = 256 * 1024 * 1024


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            block = handle.read(block_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sha256_bytes(raw):
    return hashlib.sha256(raw).hexdigest()


def sha256_json(payload):
    raw = json.dumps(payload, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()


def atomic_json(path, payload):
    temporary = Path(str(path) + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2, ensure_ascii=False)
    os.replace(temporary, path)


def frame_text(frame):
    return frame.to_csv(index=False, lineterminator="\n", float_format="%.12g")


def write_csv(path, frame):
    text = frame_text(frame)
    if path.is_file():
        assert path.read_text(encoding="utf-8") == text, f"Existing immutable CSV differs: {path}"
    else:
        path.write_text(text, encoding="utf-8")


def write_progress_csv(path, frame):
    text = frame_text(frame)
    freeze_path = FREEZE_ROOT / "Stage8B_Chest_Prediction_Freeze_Complete_v0.1.json"
    if freeze_path.is_file() or FINAL_RECORD_PATH.is_file():
        assert path.is_file() and path.read_text(encoding="utf-8") == text, f"Frozen CSV differs: {path}"
        return
    temporary = Path(str(path) + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def normalised_notebook_source_sha256(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        notebook = json.load(handle)
    payload = []
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") not in {"code", "markdown"}:
            continue
        source = cell.get("source", [])
        source = "".join(source) if isinstance(source, list) else str(source)
        payload.append({"cell_type": cell["cell_type"], "source": source.replace("\r\n", "\n")})
    return sha256_json(payload)


assert NOTEBOOK_PATH.is_file(), f"Open the frozen Drive notebook, not a renamed copy: {NOTEBOOK_PATH}"
required_stage8_paths = [
    STAGE8_FINAL_PATH, STAGE8_OUTPUT_MANIFEST_PATH, STAGE8_TBX_MANIFEST_PATH,
    STAGE8_TBX_PARTITION_PATH, STAGE8_TBX_EMBEDDING_PATH, STAGE8_TBX_AXIS_PATH,
    STAGE8_SOURCE_SUMMARY_PATH, STAGE8_EDGE_LIBRARY_PATH,
]
assert all(path.is_file() for path in required_stage8_paths)

with STAGE8_FINAL_PATH.open("r", encoding="utf-8") as handle:
    stage8_final = json.load(handle)
stage8_claim = stage8_final["final_record_sha256"]
stage8_without_claim = dict(stage8_final)
stage8_without_claim.pop("final_record_sha256")
assert sha256_json(stage8_without_claim) == stage8_claim == EXPECTED_STAGE8_FINAL_HASH
assert stage8_final["decision"].startswith("PARTIAL_CROSS_MODAL_EXPANSION")
assert stage8_final["ready_datasets_by_modality"]["chest_radiography"] == ["TBX11K"]

stage8_integrity = pd.read_csv(STAGE8_OUTPUT_MANIFEST_PATH)
assert list(stage8_integrity.columns) == ["relative_path", "size_bytes", "sha256"]
for record in stage8_integrity.itertuples(index=False):
    path = STAGE8_ROOT / record.relative_path
    assert path.is_file() and path.stat().st_size == int(record.size_bytes)
    assert sha256_file(path) == record.sha256

NLM_REGISTRY = pd.DataFrame([
    {
        "dataset": "Montgomery_CXR", "expected_images": 138,
        "index_url": "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Montgomery-County-CXR-Set/MontgomerySet/CXR_png/index.html",
        "official_provider": "US National Library of Medicine",
        "label_rule": "filename suffix _0 normal; _1 TB-consistent",
    },
    {
        "dataset": "Shenzhen_CXR", "expected_images": 662,
        "index_url": "https://data.lhncbc.nlm.nih.gov/public/Tuberculosis-Chest-X-ray-Datasets/Shenzhen-Hospital-CXR-Set/CXR_png/index.html",
        "official_provider": "US National Library of Medicine",
        "label_rule": "filename suffix _0 normal; _1 TB-consistent",
    },
])
write_csv(DATASET_REGISTRY_PATH, NLM_REGISTRY)

ANALYSIS_SPEC = {
    "scope": "append-only repair of the two NLM access failures in sealed Stage 8",
    "immutable_parent_stage": {"stage": "Stage8", "final_record_sha256": EXPECTED_STAGE8_FINAL_HASH},
    "authorised_change": "use official explicit NLM index.html endpoints instead of directory-root URLs",
    "task": "TB-consistent chest-radiograph manifestation versus non-TB",
    "datasets": ["TBX11K", "Montgomery_CXR", "Shenzhen_CXR"],
    "representation": "same fixed torchvision ResNet50 IMAGENET1K_V2 2048D L2 embedding as Stage8",
    "source_probe": "same StandardScaler plus class-balanced L2 logistic regression C=1 liblinear",
    "source_gate": "grouped development OOF and held-out validation AUC each >=0.70 with bootstrap lower 95% CI >0.55",
    "target_boundary": "freeze label-free target predictions before outcome evaluation; no target refit or tuning",
    "failure_axes": ["discrimination", "calibration", "operating_point"],
    "parent_assets_reused_without_refit": ["TBX11K embedding", "TBX11K source axis", "Stage8 retinal+skin edge library"],
    "image_storage": "NLM images streamed to memory only; none copied to Drive",
    "stage8b_drive_cap_bytes": MAXIMUM_NEW_STAGE8B_BYTES,
}
notebook_source_hash = normalised_notebook_source_sha256(NOTEBOOK_PATH)
if PROTOCOL_SEAL_PATH.is_file():
    with PROTOCOL_SEAL_PATH.open("r", encoding="utf-8") as handle:
        seal_payload = json.load(handle)
    seal_without_claim = dict(seal_payload)
    seal_claim = seal_without_claim.pop("seal_sha256")
    assert sha256_json(seal_without_claim) == seal_claim
    assert seal_payload["analysis_spec"] == ANALYSIS_SPEC
    assert seal_payload["notebook_source_sha256"] == notebook_source_hash
    assert seal_payload["parent_stage8_final_record_sha256"] == stage8_claim
else:
    seal_payload = {
        "stage": "Stage8B", "decision": "NLM_ACCESS_COMPLETION_PROTOCOL_SEALED",
        "parent_stage8_final_record_sha256": stage8_claim,
        "notebook_source_sha256": notebook_source_hash,
        "dataset_registry_sha256": sha256_file(DATASET_REGISTRY_PATH),
        "analysis_spec": ANALYSIS_SPEC, "sealed_utc": utc_now(),
    }
    seal_payload["seal_sha256"] = sha256_json(seal_payload)
    atomic_json(PROTOCOL_SEAL_PATH, seal_payload)

input_commitment = pd.DataFrame([{
    "role": path.name, "path": str(path), "size_bytes": path.stat().st_size,
    "sha256": sha256_file(path),
} for path in required_stage8_paths])
write_csv(INPUT_COMMITMENT_PATH, input_commitment)

runtime_state = {
    "stage": "Stage8B", "protocol_sealed": True,
    "parent_stage8_modified": False, "nlm_images_accessed": False,
    "images_copied_to_drive": False, "target_performance_observed_before_freeze": False,
    "target_model_refit": False, "threshold_tuned": False, "final_ddo2_fitted": False,
    "last_updated_utc": utc_now(),
}
atomic_json(RUNTIME_STATE_PATH, runtime_state)

print("Frozen Stage 8 verified:", stage8_claim)
print("Stage 8B protocol seal:", seal_payload["seal_sha256"])
print("Authorised repair: explicit official NLM index.html endpoints only")
print("Parent Stage 8 modification authorised: False")
print("Target refit / threshold tuning / final DDO2 fit: False / False / False")


Mounted at /content/drive
================ STAGE 8B NLM ACCESS-COMPLETION PREFLIGHT ================
Frozen Stage 8 verified: 698c019b8516a84522831216b6cb0c85e047e6d0b401071db99e1d74ae6b2397
Stage 8B protocol seal: 176ffac79a173dad18ad74683e8fb5348cc396f70f616324a661f7ee79816005
Authorised repair: explicit official NLM index.html endpoints only
Parent Stage 8 modification authorised: False
Target refit / threshold tuning / final DDO2 fit: False / False / False


In [2]:
#@title 08B-1. Acquire official NLM manifests through explicit index pages
import requests


BROWSER_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/png,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
    "Connection": "keep-alive",
}


def request_bytes(url, referer=None, attempts=6, timeout=(20, 180)):
    error = None
    headers = dict(BROWSER_HEADERS)
    if referer:
        headers["Referer"] = referer
    for attempt in range(attempts):
        try:
            response = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
            response.raise_for_status()
            return response.content
        except Exception as exc:
            error = exc
            time.sleep(min(2 ** attempt, 30))
    raise RuntimeError(f"Failed after {attempts} attempts: {url}: {error}")


def harmonise_nlm(dataset, index_url, expected_images):
    html = request_bytes(index_url).decode("utf-8", errors="replace")
    names = sorted(set(re.findall(r'href=["\']([^"\']+\.png)["\']', html, flags=re.I)))
    if not names:
        names = sorted(set(re.findall(r'([A-Za-z0-9_-]+_[01]\.png)', html, flags=re.I)))
    rows = []
    for name in names:
        name = Path(name).name
        match = re.search(r"_([01])\.png$", name, flags=re.I)
        if match is None:
            continue
        stem = Path(name).stem
        rows.append({
            "dataset": dataset, "modality": "chest_radiography",
            "task": "tb_manifestation_vs_non_tb", "image_id": stem,
            "unit_id": dataset + "::IMAGE::" + stem,
            "group_id": dataset + "::PATIENT::" + stem,
            "label": int(match.group(1)),
            "source_locator": urljoin(index_url, name),
            "source_index_url": index_url, "source_kind": "official_NLM_https",
        })
    frame = pd.DataFrame(rows).drop_duplicates("image_id").sort_values("image_id").reset_index(drop=True)
    counts = frame["label"].value_counts().to_dict() if len(frame) else {}
    assert len(frame) == expected_images, f"{dataset}: expected {expected_images}, found {len(frame)}"
    assert set(counts) == {0, 1} and min(counts.values()) >= 50, f"{dataset}: invalid classes {counts}"
    return frame


dataset_manifests = {}
acquisition_rows = []
for record in NLM_REGISTRY.itertuples(index=False):
    dataset = record.dataset
    manifest_path = ACQUISITION_ROOT / f"{dataset}_Harmonised_Acquisition_Manifest_v0.1.csv"
    checkpoint_path = EMBEDDING_ROOT / f"{dataset}_Frozen_ResNet50V2_L2_Embeddings_v0.1.npz"
    checkpoint_complete = False
    if checkpoint_path.is_file():
        with np.load(checkpoint_path, allow_pickle=False) as checkpoint:
            checkpoint_complete = bool(len(checkpoint["completed"]) and np.all(checkpoint["completed"].astype(np.int8) == 1))
    try:
        if manifest_path.is_file() and checkpoint_complete:
            manifest = pd.read_csv(manifest_path)
        else:
            manifest = harmonise_nlm(dataset, record.index_url, int(record.expected_images))
            write_csv(manifest_path, manifest)
        dataset_manifests[dataset] = manifest
        counts = manifest["label"].value_counts().to_dict()
        acquisition_rows.append({
            "dataset": dataset, "status": "READY", "records": len(manifest),
            "negative": int(counts.get(0, 0)), "positive": int(counts.get(1, 0)),
            "manifest_sha256": sha256_file(manifest_path), "error": "",
        })
        print(f"{dataset}: READY n={len(manifest)} classes={counts}")
    except Exception as exc:
        acquisition_rows.append({
            "dataset": dataset, "status": "UNAVAILABLE", "records": 0,
            "negative": 0, "positive": 0, "manifest_sha256": "", "error": repr(exc)[:1200],
        })
        print(f"{dataset}: UNAVAILABLE -> {exc}")

acquisition_status = pd.DataFrame(acquisition_rows)
write_progress_csv(ACQUISITION_ROOT / "Stage8B_NLM_Acquisition_Status_v0.1.csv", acquisition_status)
display(acquisition_status)
assert set(dataset_manifests) == set(NLM_REGISTRY["dataset"]), (
    "Official NLM access is not yet complete. Rerun all; no prediction freeze has been written."
)

runtime_state.update({
    "nlm_images_accessed": False,
    "ready_nlm_datasets": acquisition_status.loc[acquisition_status["status"].eq("READY"), "dataset"].tolist(),
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)


Montgomery_CXR: READY n=138 classes={0: 80, 1: 58}
Shenzhen_CXR: READY n=662 classes={1: 336, 0: 326}


,dataset,status,records,negative,positive,manifest_sha256,error
0,Montgomery_CXR,READY,138,80,58,e171358cd9bd41dae079edd94f021597de14b2421e9dca...,
1,Shenzhen_CXR,READY,662,326,336,f9b56ab870810838eef19b4bd9e608b07d31b56e0a9afb...,


In [3]:
#@title 08B-2. Extract resumable NLM embeddings and audit cross-dataset duplicates
import torch
from PIL import Image
from torch.utils.data import default_collate
from torchvision.models import ResNet50_Weights, resnet50
from tqdm.auto import tqdm


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = ResNet50_Weights.IMAGENET1K_V2
transform = weights.transforms()
backbone = resnet50(weights=weights)
backbone.fc = torch.nn.Identity()
backbone.eval().to(DEVICE)
for parameter in backbone.parameters():
    parameter.requires_grad_(False)
print("Execution device:", DEVICE)
print("Frozen representation: ResNet50 IMAGENET1K_V2, 2048D L2")


def difference_hash(image, size=16):
    gray = image.convert("L").resize((size + 1, size), Image.Resampling.BILINEAR)
    values = np.asarray(gray, dtype=np.uint8)
    bits = values[:, 1:] > values[:, :-1]
    return np.packbits(bits.reshape(-1)).tobytes().hex()


def read_nlm_image(record):
    raw = request_bytes(record["source_locator"], referer=record["source_index_url"], attempts=6)
    image = Image.open(io.BytesIO(raw)).convert("RGB")
    return transform(image), sha256_bytes(raw), difference_hash(image)


def atomic_npz(path, **arrays):
    temporary = Path(str(path) + ".tmp.npz")
    np.savez_compressed(temporary, **arrays)
    os.replace(temporary, path)


def extract_embeddings(dataset, manifest):
    checkpoint_path = EMBEDDING_ROOT / f"{dataset}_Frozen_ResNet50V2_L2_Embeddings_v0.1.npz"
    manifest_path = ACQUISITION_ROOT / f"{dataset}_Harmonised_Acquisition_Manifest_v0.1.csv"
    manifest_hash = sha256_file(manifest_path)
    ids = manifest["image_id"].astype(str).to_numpy(dtype=str)
    n = len(manifest)
    if checkpoint_path.is_file():
        saved = np.load(checkpoint_path, allow_pickle=False)
        assert str(saved["manifest_sha256"].item()) == manifest_hash
        assert np.array_equal(saved["image_id"].astype(str), ids)
        embeddings = saved["embedding"].astype(np.float32)
        completed = saved["completed"].astype(np.int8)
        content_sha = saved["content_sha256"].astype(str)
        dhash = saved["difference_hash"].astype(str)
    else:
        embeddings = np.zeros((n, FROZEN_FEATURE_DIMENSION), dtype=np.float32)
        completed = np.zeros(n, dtype=np.int8)
        content_sha = np.full(n, "", dtype="<U64")
        dhash = np.full(n, "", dtype="<U64")
    pending = np.flatnonzero(completed != 1)
    print(f"{dataset}: complete {int(np.sum(completed == 1))}/{n}; pending or retry {len(pending)}")
    for start in tqdm(range(0, len(pending), EMBEDDING_BATCH_SIZE), desc=f"Embedding {dataset}"):
        indices = pending[start:start + EMBEDDING_BATCH_SIZE]
        results = {}
        with ThreadPoolExecutor(max_workers=min(DOWNLOAD_WORKERS, len(indices))) as pool:
            futures = {pool.submit(read_nlm_image, manifest.iloc[int(index)]): int(index) for index in indices}
            for future in as_completed(futures):
                index = futures[future]
                try:
                    results[index] = future.result()
                except Exception as exc:
                    completed[index] = -1
                    print(f"Warning: {dataset}/{ids[index]} failed: {exc}")
        tensors, valid, shas, hashes = [], [], [], []
        for index in indices:
            index = int(index)
            if index in results:
                tensor, sha, image_hash = results[index]
                tensors.append(tensor); valid.append(index); shas.append(sha); hashes.append(image_hash)
        if tensors:
            with torch.inference_mode():
                features = backbone(default_collate(tensors).to(DEVICE)).float()
                features = torch.nn.functional.normalize(features, p=2, dim=1)
            features = features.cpu().numpy().astype(np.float32)
            embeddings[np.asarray(valid)] = features
            completed[np.asarray(valid)] = 1
            content_sha[np.asarray(valid)] = np.asarray(shas, dtype="<U64")
            dhash[np.asarray(valid)] = np.asarray(hashes, dtype="<U64")
        atomic_npz(
            checkpoint_path, image_id=ids, embedding=embeddings, completed=completed,
            content_sha256=content_sha, difference_hash=dhash,
            manifest_sha256=np.asarray(manifest_hash),
        )
        gc.collect()
    if not np.all(completed == 1):
        raise RuntimeError(
            f"{dataset}: {int(np.sum(completed != 1))} images remain incomplete; rerun all before freezing predictions"
        )
    return checkpoint_path


embedding_paths = {}
for dataset, manifest in dataset_manifests.items():
    try:
        embedding_paths[dataset] = extract_embeddings(dataset, manifest)
    except Exception as exc:
        print(f"Embedding failed for {dataset}: {exc}")
assert set(embedding_paths) == set(dataset_manifests), (
    "NLM embedding extraction is incomplete. Rerun all; completed checkpoints will be reused and no predictions were frozen."
)

audit_rows = []
with np.load(STAGE8_TBX_EMBEDDING_PATH, allow_pickle=False) as saved:
    tbx_manifest = pd.read_csv(STAGE8_TBX_MANIFEST_PATH)
    assert np.array_equal(tbx_manifest["image_id"].astype(str).to_numpy(), saved["image_id"].astype(str))
    for index in np.flatnonzero(saved["completed"].astype(np.int8) == 1):
        audit_rows.append({
            "dataset": "TBX11K", "row_index": int(index),
            "image_id": str(saved["image_id"][index]),
            "content_sha256": str(saved["content_sha256"][index]),
            "difference_hash": str(saved["difference_hash"][index]),
        })
for dataset, path in embedding_paths.items():
    with np.load(path, allow_pickle=False) as saved:
        for index in np.flatnonzero(saved["completed"].astype(np.int8) == 1):
            audit_rows.append({
                "dataset": dataset, "row_index": int(index),
                "image_id": str(saved["image_id"][index]),
                "content_sha256": str(saved["content_sha256"][index]),
                "difference_hash": str(saved["difference_hash"][index]),
            })
chest_duplicate_audit = pd.DataFrame(audit_rows)
if len(chest_duplicate_audit):
    sha_cross = chest_duplicate_audit.groupby("content_sha256")["dataset"].nunique()
    dhash_cross = chest_duplicate_audit.groupby("difference_hash")["dataset"].nunique()
    duplicate_sha = set(sha_cross[sha_cross > 1].index)
    duplicate_dhash = set(dhash_cross[dhash_cross > 1].index)
    chest_duplicate_audit["cross_dataset_exact_duplicate"] = chest_duplicate_audit["content_sha256"].isin(duplicate_sha)
    chest_duplicate_audit["cross_dataset_visual_hash_duplicate"] = chest_duplicate_audit["difference_hash"].isin(duplicate_dhash)
    chest_duplicate_audit["excluded_for_cross_dataset_duplicate"] = (
        chest_duplicate_audit["cross_dataset_exact_duplicate"] |
        chest_duplicate_audit["cross_dataset_visual_hash_duplicate"]
    )
else:
    chest_duplicate_audit = pd.DataFrame(columns=[
        "dataset", "row_index", "image_id", "content_sha256", "difference_hash",
        "cross_dataset_exact_duplicate", "cross_dataset_visual_hash_duplicate",
        "excluded_for_cross_dataset_duplicate",
    ])
write_progress_csv(EMBEDDING_ROOT / "Stage8B_Chest_CrossDataset_Duplicate_Audit_v0.1.csv", chest_duplicate_audit)
print("Completed new NLM embeddings:", sorted(embedding_paths))
print("Cross-dataset duplicate rows excluded:", int(chest_duplicate_audit["excluded_for_cross_dataset_duplicate"].sum()) if len(chest_duplicate_audit) else 0)

runtime_state.update({
    "nlm_images_accessed": bool(len(embedding_paths)), "images_copied_to_drive": False,
    "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]


Execution device: cuda
Frozen representation: ResNet50 IMAGENET1K_V2, 2048D L2
Montgomery_CXR: complete 0/138; pending or retry 138


Embedding Montgomery_CXR:   0%|          | 0/9 [00:00<?, ?it/s]

Shenzhen_CXR: complete 0/662; pending or retry 662


Embedding Shenzhen_CXR:   0%|          | 0/42 [00:00<?, ?it/s]

Completed new NLM embeddings: ['Montgomery_CXR', 'Shenzhen_CXR']
Cross-dataset duplicate rows excluded: 0


In [4]:
#@title 08B-3. Reuse the frozen TBX11K axis and fit fixed NLM source axes
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def aggregate_records(manifest, embeddings):
    work = manifest.copy().reset_index(drop=True)
    work["embedding_index"] = np.arange(len(work))
    records, vectors = [], []
    for unit_id, group in work.groupby("unit_id", sort=True):
        labels = group["label"].astype(int).unique()
        if len(labels) != 1:
            continue
        vector = embeddings[group["embedding_index"].to_numpy(int)].mean(axis=0)
        norm = np.linalg.norm(vector)
        if not np.isfinite(norm) or norm == 0:
            continue
        records.append({
            "unit_id": str(unit_id), "group_id": sorted(group["group_id"].astype(str).unique())[0],
            "label": int(labels[0]), "images": int(len(group)),
        })
        vectors.append((vector / norm).astype(np.float32))
    return pd.DataFrame(records), np.asarray(vectors, dtype=np.float32)


def fixed_group_split(table, seed):
    test_size = 0.40 if len(table) < 300 else 0.25
    splitter = GroupShuffleSplit(n_splits=64, test_size=test_size, random_state=seed)
    for development, validation in splitter.split(table, table["label"], table["group_id"]):
        if table.iloc[development]["label"].nunique() == 2 and table.iloc[validation]["label"].nunique() == 2:
            return np.asarray(development), np.asarray(validation)
    raise ValueError("Could not create a two-class group-disjoint split")


def create_probe():
    return Pipeline([
        ("standardscaler", StandardScaler()),
        ("logisticregression", LogisticRegression(
            C=1.0, penalty="l2", class_weight="balanced", solver="liblinear",
            max_iter=5000, random_state=RANDOM_SEED,
        )),
    ])


def patient_bootstrap_auc(table, probabilities, seed):
    labels = table["label"].to_numpy(int)
    probabilities = np.asarray(probabilities, dtype=float)
    groups = table["group_id"].astype(str).to_numpy()
    unique_groups = np.unique(groups)
    rng = np.random.default_rng(seed)
    values = []
    attempts = 0
    while len(values) < N_BOOTSTRAP and attempts < N_BOOTSTRAP * 20:
        attempts += 1
        sampled = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        indices = np.concatenate([np.flatnonzero(groups == group) for group in sampled])
        if np.unique(labels[indices]).size == 2:
            values.append(roc_auc_score(labels[indices], probabilities[indices]))
    if len(values) < N_BOOTSTRAP // 2:
        return [np.nan, np.nan]
    return [float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))]


def load_axis(path):
    with np.load(path, allow_pickle=False) as saved:
        return {
            "mean": saved["scaler_mean"].astype(np.float64),
            "scale": saved["scaler_scale"].astype(np.float64),
            "standardised_coefficient": saved["standardised_coefficient"].astype(np.float64),
            "standardised_intercept": float(saved["standardised_intercept"].item()),
            "raw_coefficient": saved["raw_coefficient"].astype(np.float64),
            "raw_intercept": float(saved["raw_intercept"].item()),
        }


def save_axis(path, probe, development_manifest_hash, source):
    scaler = probe.named_steps["standardscaler"]
    logistic = probe.named_steps["logisticregression"]
    coefficient = logistic.coef_.reshape(-1).astype(np.float64)
    intercept = float(logistic.intercept_[0])
    mean = scaler.mean_.astype(np.float64)
    scale = scaler.scale_.astype(np.float64)
    raw_coefficient = coefficient / scale
    raw_intercept = intercept - float(np.dot(mean / scale, coefficient))
    candidate = {
        "mean": mean, "scale": scale, "standardised_coefficient": coefficient,
        "standardised_intercept": intercept, "raw_coefficient": raw_coefficient,
        "raw_intercept": raw_intercept,
    }
    if path.is_file():
        with np.load(path, allow_pickle=False) as saved:
            assert str(saved["source"].item()) == source
            assert str(saved["development_manifest_sha256"].item()) == development_manifest_hash
        existing = load_axis(path)
        for key in candidate:
            assert np.allclose(existing[key], candidate[key], rtol=1e-10, atol=1e-12)
        return existing
    atomic_npz(
        path, source=np.asarray(source), scaler_mean=mean, scaler_scale=scale,
        standardised_coefficient=coefficient, standardised_intercept=np.asarray(intercept),
        raw_coefficient=raw_coefficient, raw_intercept=np.asarray(raw_intercept),
        development_manifest_sha256=np.asarray(development_manifest_hash),
    )
    return candidate


def axis_probability(axis, embeddings):
    embeddings = np.asarray(embeddings, dtype=np.float64)
    standardised_logit = ((embeddings - axis["mean"]) / axis["scale"]) @ axis["standardised_coefficient"] + axis["standardised_intercept"]
    raw_logit = embeddings @ axis["raw_coefficient"] + axis["raw_intercept"]
    error = float(np.max(np.abs(standardised_logit - raw_logit)))
    assert error < 1e-8, f"Axis equivalence failure: {error}"
    return expit(raw_logit), raw_logit, error


duplicate_lookup = set()
if len(chest_duplicate_audit):
    excluded = chest_duplicate_audit[chest_duplicate_audit["excluded_for_cross_dataset_duplicate"]]
    duplicate_lookup = set(zip(excluded["dataset"], excluded["row_index"].astype(int)))

domain_assets = {}
inventory_rows = []

# Reconstruct TBX11K units from sealed Stage 8 embeddings and reuse its frozen partition.
with np.load(STAGE8_TBX_EMBEDDING_PATH, allow_pickle=False) as saved:
    tbx_manifest = pd.read_csv(STAGE8_TBX_MANIFEST_PATH).reset_index(drop=True)
    valid = [
        int(index) for index in np.flatnonzero(saved["completed"].astype(np.int8) == 1)
        if ("TBX11K", int(index)) not in duplicate_lookup
    ]
    tbx_table, tbx_vectors = aggregate_records(tbx_manifest.iloc[valid].reset_index(drop=True), saved["embedding"][valid])
tbx_partition = pd.read_csv(STAGE8_TBX_PARTITION_PATH)[["unit_id", "partition"]]
tbx_table = tbx_table.merge(tbx_partition, on="unit_id", how="left", validate="one_to_one")
assert tbx_table["partition"].isin(["development", "validation"]).all()
domain_assets["TBX11K"] = {"table": tbx_table, "embedding": tbx_vectors}

for dataset, checkpoint_path in embedding_paths.items():
    with np.load(checkpoint_path, allow_pickle=False) as saved:
        manifest = dataset_manifests[dataset].reset_index(drop=True)
        valid = [
            int(index) for index in np.flatnonzero(saved["completed"].astype(np.int8) == 1)
            if (dataset, int(index)) not in duplicate_lookup
        ]
        if len(valid) < 80:
            continue
        table, vectors = aggregate_records(manifest.iloc[valid].reset_index(drop=True), saved["embedding"][valid])
    if len(table) < 80 or table["label"].nunique() != 2:
        continue
    development, validation = fixed_group_split(table, RANDOM_SEED + len(domain_assets))
    table["partition"] = ""
    table.loc[development, "partition"] = "development"
    table.loc[validation, "partition"] = "validation"
    assert set(table["partition"]) == {"development", "validation"}
    domain_assets[dataset] = {"table": table, "embedding": vectors}

for dataset, asset in domain_assets.items():
    for partition in ["development", "validation"]:
        subset = asset["table"][asset["table"]["partition"].eq(partition)]
        inventory_rows.append({
            "dataset": dataset, "partition": partition, "units": len(subset),
            "groups": subset["group_id"].nunique(), "negative": int((subset["label"] == 0).sum()),
            "positive": int((subset["label"] == 1).sum()),
        })
partition_inventory = pd.DataFrame(inventory_rows)
write_progress_csv(ACQUISITION_ROOT / "Stage8B_Chest_Partition_Inventory_v0.1.csv", partition_inventory)
display(partition_inventory)

axes = {"TBX11K": load_axis(STAGE8_TBX_AXIS_PATH)}
source_validation_assets = {}
source_rows = []
stage8_source_summary = pd.read_csv(STAGE8_SOURCE_SUMMARY_PATH).set_index("source")
tbx_asset = domain_assets["TBX11K"]
tbx_validation_mask = tbx_asset["table"]["partition"].eq("validation").to_numpy()
tbx_validation = tbx_asset["table"].loc[tbx_validation_mask].reset_index(drop=True).copy()
tbx_probability, tbx_logit, tbx_error = axis_probability(axes["TBX11K"], tbx_asset["embedding"][tbx_validation_mask])
tbx_validation["probability"] = tbx_probability
tbx_validation["logit"] = tbx_logit
source_validation_assets["TBX11K"] = tbx_validation
old_tbx = stage8_source_summary.loc["TBX11K"]
source_rows.append({
    "source": "TBX11K", "development_units": int(old_tbx["development_units"]),
    "validation_units": int(old_tbx["validation_units"]),
    "development_oof_auc": float(old_tbx["development_oof_auc"]),
    "development_oof_auc_ci_lower": float(old_tbx["development_oof_auc_ci_lower"]),
    "development_oof_auc_ci_upper": float(old_tbx["development_oof_auc_ci_upper"]),
    "validation_auc": float(old_tbx["validation_auc"]),
    "validation_auc_ci_lower": float(old_tbx["validation_auc_ci_lower"]),
    "validation_auc_ci_upper": float(old_tbx["validation_auc_ci_upper"]),
    "recoverable": bool(old_tbx["recoverable"]), "failure_reason": "",
    "axis_provenance": "REUSED_SEALED_STAGE8_WITHOUT_REFIT",
    "axis_path": str(STAGE8_TBX_AXIS_PATH), "axis_sha256": sha256_file(STAGE8_TBX_AXIS_PATH),
    "maximum_axis_equivalence_error": tbx_error,
})

for source_index, source in enumerate(sorted(set(domain_assets) - {"TBX11K"}), 1):
    asset = domain_assets[source]
    table = asset["table"]
    embeddings = asset["embedding"].astype(np.float64)
    development_mask = table["partition"].eq("development").to_numpy()
    validation_mask = table["partition"].eq("validation").to_numpy()
    development_table = table.loc[development_mask].reset_index(drop=True)
    validation_table = table.loc[validation_mask].reset_index(drop=True)
    X_development = embeddings[development_mask]
    y_development = development_table["label"].to_numpy(int)
    groups = development_table["group_id"].astype(str).to_numpy()
    folds = min(5, np.unique(groups[y_development == 1]).size, np.unique(groups[y_development == 0]).size)
    if folds < 3:
        source_rows.append({
            "source": source, "development_units": len(development_table), "validation_units": len(validation_table),
            "development_oof_auc": np.nan, "development_oof_auc_ci_lower": np.nan, "development_oof_auc_ci_upper": np.nan,
            "validation_auc": np.nan, "validation_auc_ci_lower": np.nan, "validation_auc_ci_upper": np.nan,
            "recoverable": False, "failure_reason": "INSUFFICIENT_GROUPS_FOR_THREE_FOLDS",
            "axis_provenance": "NOT_FIT", "axis_path": "", "axis_sha256": "",
            "maximum_axis_equivalence_error": np.nan,
        })
        continue
    splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=RANDOM_SEED)
    oof = np.full(len(development_table), np.nan)
    for training, testing in splitter.split(X_development, y_development, groups):
        probe = create_probe()
        probe.fit(X_development[training], y_development[training])
        oof[testing] = probe.predict_proba(X_development[testing])[:, 1]
    assert np.isfinite(oof).all()
    oof_auc = float(roc_auc_score(y_development, oof))
    oof_ci = patient_bootstrap_auc(development_table, oof, RANDOM_SEED + source_index)
    final_probe = create_probe()
    final_probe.fit(X_development, y_development)
    partition_path = ACQUISITION_ROOT / f"{source}_Frozen_Development_Validation_Unit_Manifest_v0.1.csv"
    write_csv(partition_path, table)
    axis_path = AXIS_ROOT / f"{source}_Frozen_Source_Axis_v0.1.npz"
    axis = save_axis(axis_path, final_probe, sha256_file(partition_path), source)
    probabilities, logits, equivalence_error = axis_probability(axis, embeddings[validation_mask])
    validation_auc = float(roc_auc_score(validation_table["label"], probabilities))
    validation_ci = patient_bootstrap_auc(validation_table, probabilities, RANDOM_SEED + 100 + source_index)
    recoverable = bool(
        oof_auc >= AUC_MINIMUM and oof_ci[0] > AUC_CI_LOWER_STRICT_MINIMUM and
        validation_auc >= AUC_MINIMUM and validation_ci[0] > AUC_CI_LOWER_STRICT_MINIMUM
    )
    axes[source] = axis
    validation_table = validation_table.copy()
    validation_table["probability"] = probabilities
    validation_table["logit"] = logits
    source_validation_assets[source] = validation_table
    source_rows.append({
        "source": source, "development_units": len(development_table), "validation_units": len(validation_table),
        "development_oof_auc": oof_auc, "development_oof_auc_ci_lower": oof_ci[0],
        "development_oof_auc_ci_upper": oof_ci[1], "validation_auc": validation_auc,
        "validation_auc_ci_lower": validation_ci[0], "validation_auc_ci_upper": validation_ci[1],
        "recoverable": recoverable, "failure_reason": "" if recoverable else "SOURCE_GATE_NOT_MET",
        "axis_provenance": "FIT_ON_STAGE8B_NLM_DEVELOPMENT_ONLY",
        "axis_path": str(axis_path.relative_to(STAGE8B_ROOT)), "axis_sha256": sha256_file(axis_path),
        "maximum_axis_equivalence_error": equivalence_error,
    })
    print(f"{source}: OOF {oof_auc:.4f} [{oof_ci[0]:.4f}, {oof_ci[1]:.4f}] | validation {validation_auc:.4f} [{validation_ci[0]:.4f}, {validation_ci[1]:.4f}] | recoverable={recoverable}")

source_columns = [
    "source", "development_units", "validation_units", "development_oof_auc",
    "development_oof_auc_ci_lower", "development_oof_auc_ci_upper", "validation_auc",
    "validation_auc_ci_lower", "validation_auc_ci_upper", "recoverable", "failure_reason",
    "axis_provenance", "axis_path", "axis_sha256", "maximum_axis_equivalence_error",
]
source_recoverability = pd.DataFrame(source_rows).reindex(columns=source_columns)
write_progress_csv(AXIS_ROOT / "Stage8B_Chest_Source_Recoverability_Summary_v0.1.csv", source_recoverability)
display(source_recoverability)


,dataset,partition,units,groups,negative,positive
0,TBX11K,development,1500,1500,1374,126
1,TBX11K,validation,500,500,453,47
2,Montgomery_CXR,development,82,82,47,35
3,Montgomery_CXR,validation,56,56,33,23
4,Shenzhen_CXR,development,496,496,239,257
5,Shenzhen_CXR,validation,166,166,87,79


Montgomery_CXR: OOF 0.7775 [0.6716, 0.8786] | validation 0.8353 [0.7220, 0.9307] | recoverable=True
Shenzhen_CXR: OOF 0.8716 [0.8371, 0.8992] | validation 0.8709 [0.8129, 0.9209] | recoverable=True


,source,development_units,validation_units,development_oof_auc,development_oof_auc_ci_lower,development_oof_auc_ci_upper,validation_auc,validation_auc_ci_lower,validation_auc_ci_upper,recoverable,failure_reason,axis_provenance,axis_path,axis_sha256,maximum_axis_equivalence_error
0,TBX11K,1500,500,0.958827,0.940773,0.975102,0.963788,0.939730,0.983316,True,,REUSED_SEALED_STAGE8_WITHOUT_REFIT,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,6fc63f746731f607173d7aff4ffdfb2e580be2a4abe5b6...,1.643130e-14
1,Montgomery_CXR,82,56,0.777508,0.671599,0.878600,0.835310,0.721985,0.930683,True,,FIT_ON_STAGE8B_NLM_DEVELOPMENT_ONLY,03_Frozen_Source_Axes/Montgomery_CXR_Frozen_So...,aed9f440093253886d85a11134bba078c195ca6f688f85...,8.881784e-15
2,Shenzhen_CXR,496,166,0.871595,0.837057,0.899187,0.870944,0.812921,0.920921,True,,FIT_ON_STAGE8B_NLM_DEVELOPMENT_ONLY,03_Frozen_Source_Axes/Shenzhen_CXR_Frozen_Sour...,11ebbbc4d42c5fe29d3ade72e0350ef681623862c6f3e1...,1.243450e-14


In [5]:
#@title 08B-4. Score chest targets label-free and freeze predictions
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from scipy.stats import wasserstein_distance
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.neighbors import NearestNeighbors


def deterministic_indices(ids, limit, salt):
    ranked = sorted(range(len(ids)), key=lambda index: hashlib.sha256(f"{salt}|{ids[index]}".encode()).hexdigest())
    return np.asarray(ranked[:min(limit, len(ranked))], dtype=int)


def support_components(source_embeddings, target_embeddings):
    neighbors = min(6, len(source_embeddings))
    model = NearestNeighbors(n_neighbors=neighbors, metric="cosine").fit(source_embeddings)
    source_distances = model.kneighbors(source_embeddings, return_distance=True)[0][:, 1:].mean(axis=1)
    target_distances = model.kneighbors(target_embeddings, n_neighbors=min(5, len(source_embeddings)), return_distance=True)[0].mean(axis=1)
    q95, q99 = np.quantile(source_distances, [0.95, 0.99])
    median = float(np.median(source_distances))
    return {
        "support_fraction": float(np.mean(target_distances <= q95)),
        "fraction_beyond_source_q99": float(np.mean(target_distances > q99)),
        "source_knn_q95": float(q95), "source_knn_q99": float(q99),
        "target_mean_knn_distance": float(target_distances.mean()),
        "target_mean_knn_distance_normalised": float(target_distances.mean() / max(median, 1e-8)),
    }


def geometry_components(source_table, source_embeddings, target_table, target_embeddings, salt):
    limit = min(128, len(source_table), len(target_table))
    source_index = deterministic_indices(source_table["unit_id"].tolist(), limit, salt + "|S")
    target_index = deterministic_indices(target_table["unit_id"].tolist(), limit, salt + "|T")
    source_sample = source_embeddings[source_index].astype(float)
    target_sample = target_embeddings[target_index].astype(float)
    pooled = np.concatenate([source_sample, target_sample], axis=0)
    domain_labels = np.concatenate([np.zeros(limit, dtype=int), np.ones(limit, dtype=int)])
    components = min(32, pooled.shape[0] - 2, pooled.shape[1])
    projected = PCA(n_components=components, svd_solver="randomized", random_state=RANDOM_SEED).fit_transform(pooled)
    classifier = Pipeline([
        ("scale", StandardScaler()),
        ("logistic", LogisticRegression(C=1.0, class_weight="balanced", solver="liblinear", random_state=RANDOM_SEED)),
    ])
    probability = cross_val_predict(
        classifier, projected, domain_labels,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED),
        method="predict_proba",
    )[:, 1]
    domain_auc = float(roc_auc_score(domain_labels, probability))
    squared = cdist(source_sample, source_sample, metric="sqeuclidean")
    positive = squared[np.triu_indices_from(squared, k=1)]
    bandwidth = max(float(np.median(positive[positive > 0])), 1e-8)
    kss = np.exp(-cdist(source_sample, source_sample, metric="sqeuclidean") / bandwidth).mean()
    ktt = np.exp(-cdist(target_sample, target_sample, metric="sqeuclidean") / bandwidth).mean()
    kst = np.exp(-cdist(source_sample, target_sample, metric="sqeuclidean") / bandwidth).mean()
    cost = cdist(source_sample, target_sample, metric="cosine")
    rows, columns = linear_sum_assignment(cost)
    return {"domain_auc": domain_auc, "rbf_mmd2": float(kss + ktt - 2 * kst), "ot_cosine_cost": float(cost[rows, columns].mean())}


def mixture_components(source_table, target_logits, source_iqr):
    negative = source_table.loc[source_table["label"].eq(0), "logit"].to_numpy(float)
    positive = source_table.loc[source_table["label"].eq(1), "logit"].to_numpy(float)
    values = np.concatenate([negative, positive])
    best = None
    for prevalence in np.linspace(0, 1, 101):
        weights = np.concatenate([
            np.full(len(negative), (1 - prevalence) / len(negative)),
            np.full(len(positive), prevalence / len(positive)),
        ])
        distance = wasserstein_distance(
            np.asarray(target_logits, float), values,
            u_weights=np.full(len(target_logits), 1 / len(target_logits)), v_weights=weights,
        )
        if best is None or distance < best[0]:
            best = (float(distance), float(prevalence))
    return {
        "mixture_wasserstein_residual_normalised": float(best[0] / max(source_iqr, 1e-8)),
        "unlabeled_mixture_prevalence": best[1],
    }


def source_calibration_reference(table):
    probabilities = np.clip(table["probability"].to_numpy(float), 1e-12, 1 - 1e-12)
    labels = table["label"].to_numpy(int)
    error_rate = 1 - float(np.mean((probabilities >= FIXED_THRESHOLD).astype(int) == labels))
    confidence = np.maximum(probabilities, 1 - probabilities)
    return float(np.quantile(confidence, error_rate))


label_free_rows, prediction_rows = [], []
recoverable_sources = source_recoverability.loc[source_recoverability["recoverable"].fillna(False), "source"].tolist()
source_indexed = source_recoverability.set_index("source")
for source in recoverable_sources:
    source_asset = domain_assets[source]
    development_mask = source_asset["table"]["partition"].eq("development").to_numpy()
    source_development = source_asset["table"].loc[development_mask].reset_index(drop=True).copy()
    source_vectors = source_asset["embedding"][development_mask]
    source_probability, source_logits, _ = axis_probability(axes[source], source_vectors)
    source_development["probability"] = source_probability
    source_development["logit"] = source_logits
    source_validation = source_validation_assets[source]
    source_iqr = float(np.subtract(*np.quantile(source_validation["logit"], [0.75, 0.25])))
    atc_threshold = source_calibration_reference(source_validation)
    for target, target_asset in domain_assets.items():
        if target == source:
            continue
        target_mask = target_asset["table"]["partition"].eq("validation").to_numpy()
        target_table = target_asset["table"].loc[target_mask].reset_index(drop=True).copy()
        target_vectors = target_asset["embedding"][target_mask]
        probabilities, logits, equivalence_error = axis_probability(axes[source], target_vectors)
        support = support_components(source_vectors, target_vectors)
        geometry = geometry_components(source_development, source_vectors, target_table, target_vectors, f"{source}|{target}")
        mixture = mixture_components(source_development, logits, source_iqr)
        confidence = np.maximum(probabilities, 1 - probabilities)
        entropy = -(probabilities * np.log(np.clip(probabilities, 1e-12, 1)) + (1 - probabilities) * np.log(np.clip(1 - probabilities, 1e-12, 1)))
        target_iqr = float(np.subtract(*np.quantile(logits, [0.75, 0.25])))
        edge_id = f"{source}__TO__{target}"
        label_free_rows.append({
            "edge_id": edge_id, "modality": "chest_radiography", "task": "tb_manifestation_vs_non_tb",
            "source": source, "target": target, "target_units": len(target_table),
            "source_validation_auc": float(source_indexed.loc[source, "validation_auc"]),
            "mean_confidence": float(confidence.mean()), "mean_entropy_nats": float(entropy.mean()),
            "atc_estimated_accuracy": float(np.mean(confidence >= atc_threshold)),
            "atc_threshold_from_source_validation": atc_threshold,
            "target_to_source_logit_iqr_ratio": float(target_iqr / max(source_iqr, 1e-8)),
            "maximum_axis_equivalence_error": equivalence_error, **support, **geometry, **mixture,
        })
        for index, record in target_table.iterrows():
            prediction_rows.append({
                "edge_id": edge_id, "modality": "chest_radiography", "task": "tb_manifestation_vs_non_tb",
                "source": source, "target": target, "unit_id": record["unit_id"], "group_id": record["group_id"],
                "probability": float(probabilities[index]), "logit": float(logits[index]), "images": int(record["images"]),
            })

label_free_columns = [
    "edge_id", "modality", "task", "source", "target", "target_units", "source_validation_auc",
    "mean_confidence", "mean_entropy_nats", "atc_estimated_accuracy", "atc_threshold_from_source_validation",
    "target_to_source_logit_iqr_ratio", "maximum_axis_equivalence_error", "support_fraction",
    "fraction_beyond_source_q99", "source_knn_q95", "source_knn_q99", "target_mean_knn_distance",
    "target_mean_knn_distance_normalised", "domain_auc", "rbf_mmd2", "ot_cosine_cost",
    "mixture_wasserstein_residual_normalised", "unlabeled_mixture_prevalence",
]
prediction_columns = ["edge_id", "modality", "task", "source", "target", "unit_id", "group_id", "probability", "logit", "images"]
label_free_edges = pd.DataFrame(label_free_rows).reindex(columns=label_free_columns)
frozen_predictions = pd.DataFrame(prediction_rows).reindex(columns=prediction_columns)
LABEL_FREE_PATH = FREEZE_ROOT / "Stage8B_LabelFree_Chest_Edge_Components_v0.1.csv"
PREDICTION_PATH = FREEZE_ROOT / "Stage8B_LabelFree_Chest_Unit_Predictions_v0.1.csv"
write_csv(LABEL_FREE_PATH, label_free_edges)
write_csv(PREDICTION_PATH, frozen_predictions)

freeze_payload = {
    "stage": "Stage8B", "decision": "CHEST_TARGET_PREDICTIONS_FROZEN_BEFORE_OUTCOME_EVALUATION",
    "parent_stage8_final_record_sha256": stage8_claim, "protocol_seal_sha256": seal_payload["seal_sha256"],
    "recoverable_sources": recoverable_sources, "frozen_edges": int(len(label_free_edges)),
    "label_free_components_sha256": sha256_file(LABEL_FREE_PATH),
    "unit_predictions_sha256": sha256_file(PREDICTION_PATH),
    "labels_in_prediction_files": False, "target_model_refit": False,
    "threshold_tuned": False, "final_ddo2_fitted": False, "frozen_utc": seal_payload["sealed_utc"],
}
freeze_payload["freeze_sha256"] = sha256_json(freeze_payload)
PREDICTION_FREEZE_PATH = FREEZE_ROOT / "Stage8B_Chest_Prediction_Freeze_Complete_v0.1.json"
if PREDICTION_FREEZE_PATH.is_file():
    with PREDICTION_FREEZE_PATH.open("r", encoding="utf-8") as handle:
        assert json.load(handle) == freeze_payload
else:
    atomic_json(PREDICTION_FREEZE_PATH, freeze_payload)
print("Recoverable chest sources:", recoverable_sources)
print("Frozen chest edges:", len(label_free_edges))
print("Prediction freeze hash:", freeze_payload["freeze_sha256"])
print("Target performance observed before freeze: False")
display(label_free_edges)


Recoverable chest sources: ['TBX11K', 'Montgomery_CXR', 'Shenzhen_CXR']
Frozen chest edges: 6
Prediction freeze hash: c0fb835e26f7ad23d6132e3e72603e110a92a51d50ecdb19c4546278fb1ee020
Target performance observed before freeze: False


,edge_id,modality,task,source,target,target_units,source_validation_auc,mean_confidence,mean_entropy_nats,atc_estimated_accuracy,...,fraction_beyond_source_q99,source_knn_q95,source_knn_q99,target_mean_knn_distance,target_mean_knn_distance_normalised,domain_auc,rbf_mmd2,ot_cosine_cost,mixture_wasserstein_residual_normalised,unlabeled_mixture_prevalence
0,TBX11K__TO__Montgomery_CXR,chest_radiography,tb_manifestation_vs_non_tb,TBX11K,Montgomery_CXR,56,0.963788,0.888677,0.264665,0.892857,...,0.285714,0.179299,0.215495,0.196371,1.591404,1.000000,0.214320,0.262554,0.779409,0.18
1,TBX11K__TO__Shenzhen_CXR,chest_radiography,tb_manifestation_vs_non_tb,TBX11K,Shenzhen_CXR,166,0.963788,0.856633,0.323837,0.813253,...,0.000000,0.179299,0.215495,0.146241,1.185147,0.981873,0.109379,0.198461,1.009593,0.30
2,Montgomery_CXR__TO__TBX11K,chest_radiography,tb_manifestation_vs_non_tb,Montgomery_CXR,TBX11K,500,0.835310,0.919978,0.185179,0.816000,...,0.794000,0.181283,0.193720,0.231863,1.803218,0.999851,0.229077,0.262486,0.479484,0.72
3,Montgomery_CXR__TO__Shenzhen_CXR,chest_radiography,tb_manifestation_vs_non_tb,Montgomery_CXR,Shenzhen_CXR,166,0.835310,0.934381,0.151162,0.849398,...,0.993976,0.181283,0.193720,0.289379,2.250530,1.000000,0.383563,0.336612,0.358290,0.89
4,Shenzhen_CXR__TO__TBX11K,chest_radiography,tb_manifestation_vs_non_tb,Shenzhen_CXR,TBX11K,500,0.870944,0.908227,0.212207,0.790000,...,0.074000,0.180022,0.231318,0.173601,1.434847,0.987854,0.110576,0.204258,0.321828,0.43
5,Shenzhen_CXR__TO__Montgomery_CXR,chest_radiography,tb_manifestation_vs_non_tb,Shenzhen_CXR,Montgomery_CXR,56,0.870944,0.882863,0.242866,0.696429,...,0.642857,0.180022,0.231318,0.257371,2.127219,1.000000,0.353827,0.339794,0.348595,0.63


In [6]:
#@title 08B-5. Evaluate frozen chest predictions and extend the evidence library
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, brier_score_loss, confusion_matrix, log_loss


def calibration_error(labels, probabilities, bins=10):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    order = np.argsort(probabilities, kind="mergesort")
    bin_ids = np.empty(len(labels), dtype=int)
    bin_ids[order] = np.minimum(np.floor(np.arange(len(labels)) * bins / len(labels)).astype(int), bins - 1)
    value = 0.0
    for bin_id in range(bins):
        mask = bin_ids == bin_id
        if np.any(mask):
            value += mask.mean() * abs(probabilities[mask].mean() - labels[mask].mean())
    return float(value)


def outcome_metrics(labels, probabilities):
    labels = np.asarray(labels, dtype=int)
    probabilities = np.clip(np.asarray(probabilities, dtype=float), 1e-12, 1 - 1e-12)
    predictions = (probabilities >= FIXED_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0, 1]).ravel()
    sensitivity = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    return {
        "auc": float(roc_auc_score(labels, probabilities)),
        "average_precision": float(average_precision_score(labels, probabilities)),
        "brier": float(brier_score_loss(labels, probabilities)),
        "log_loss": float(log_loss(labels, probabilities, labels=[0, 1])),
        "ece10": calibration_error(labels, probabilities),
        "balanced_accuracy": float((sensitivity + specificity) / 2),
        "sensitivity": float(sensitivity), "specificity": float(specificity),
    }


label_lookup = {}
for dataset, asset in domain_assets.items():
    validation = asset["table"][asset["table"]["partition"].eq("validation")]
    label_lookup[dataset] = validation.set_index("unit_id")[["label", "group_id"]]

edge_rows, evaluated_predictions = [], []
for edge in label_free_edges.itertuples(index=False):
    target_predictions = frozen_predictions[frozen_predictions["edge_id"].eq(edge.edge_id)].copy()
    target_predictions = target_predictions.join(label_lookup[edge.target][["label"]], on="unit_id", how="inner")
    if target_predictions["label"].nunique() != 2:
        continue
    metrics = outcome_metrics(target_predictions["label"], target_predictions["probability"])
    interval = patient_bootstrap_auc(
        target_predictions[["unit_id", "group_id", "label"]],
        target_predictions["probability"].to_numpy(), RANDOM_SEED + len(edge_rows) + 500,
    )
    source_validation = source_validation_assets[edge.source]
    source_metrics = outcome_metrics(source_validation["label"], source_validation["probability"])
    discrimination_pass = bool(metrics["auc"] >= AUC_MINIMUM and interval[0] > AUC_CI_LOWER_STRICT_MINIMUM)
    calibration_pass = bool(
        metrics["ece10"] - source_metrics["ece10"] <= CALIBRATION_DEGRADATION_TOLERANCE and
        metrics["brier"] - source_metrics["brier"] <= CALIBRATION_DEGRADATION_TOLERANCE
    )
    operating_pass = bool(metrics["balanced_accuracy"] >= OPERATING_POINT_BALANCED_ACCURACY_MINIMUM)
    if discrimination_pass and calibration_pass and operating_pass:
        state = "DISCRIMINATION_CALIBRATION_AND_OPERATING_POINT_RETAINED"
    elif discrimination_pass and not calibration_pass:
        state = "DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED"
    elif discrimination_pass and calibration_pass and not operating_pass:
        state = "DISCRIMINATION_AND_CALIBRATION_RETAINED_BUT_OPERATING_POINT_FAILED"
    else:
        state = "DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIONAL_FAILURES"
    row = label_free_edges[label_free_edges["edge_id"].eq(edge.edge_id)].iloc[0].to_dict()
    row.update({
        "source_recoverable": True, "target_auc": metrics["auc"],
        "target_auc_ci_lower": interval[0], "target_auc_ci_upper": interval[1],
        "source_minus_target_auc": source_metrics["auc"] - metrics["auc"],
        "target_average_precision": metrics["average_precision"], "target_brier": metrics["brier"],
        "target_ece10": metrics["ece10"], "calibration_ece_degradation": metrics["ece10"] - source_metrics["ece10"],
        "calibration_brier_degradation": metrics["brier"] - source_metrics["brier"],
        "target_balanced_accuracy_at_0_5": metrics["balanced_accuracy"],
        "target_sensitivity_at_0_5": metrics["sensitivity"], "target_specificity_at_0_5": metrics["specificity"],
        "discrimination_pass": discrimination_pass, "calibration_pass": calibration_pass,
        "operating_point_pass": operating_pass, "observed_transportability_state": state,
    })
    edge_rows.append(row)
    evaluated_predictions.append(target_predictions)

edge_columns = list(label_free_edges.columns) + [
    "source_recoverable", "target_auc", "target_auc_ci_lower", "target_auc_ci_upper",
    "source_minus_target_auc", "target_average_precision", "target_brier", "target_ece10",
    "calibration_ece_degradation", "calibration_brier_degradation",
    "target_balanced_accuracy_at_0_5", "target_sensitivity_at_0_5", "target_specificity_at_0_5",
    "discrimination_pass", "calibration_pass", "operating_point_pass", "observed_transportability_state",
]
chest_edge_matrix = pd.DataFrame(edge_rows).reindex(columns=list(dict.fromkeys(edge_columns)))
evaluated_unit_predictions = pd.concat(evaluated_predictions, ignore_index=True) if evaluated_predictions else pd.DataFrame()
write_csv(DISCOVERY_ROOT / "Stage8B_Chest_Edge_Matrix_v0.1.csv", chest_edge_matrix)
write_csv(DISCOVERY_ROOT / "Stage8B_Evaluated_Chest_Unit_Predictions_v0.1.csv", evaluated_unit_predictions)

relations = [
    ("target_auc", "target_mean_knn_distance", -1),
    ("source_minus_target_auc", "atc_estimated_accuracy", 1),
    ("target_ece10", "unlabeled_mixture_prevalence", -1),
    ("calibration_ece_degradation", "atc_estimated_accuracy", 1),
    ("target_balanced_accuracy_at_0_5", "unlabeled_mixture_prevalence", 1),
]
replication_rows = []
for outcome, component, expected_sign in relations:
    subset = chest_edge_matrix[[component, outcome]].replace([np.inf, -np.inf], np.nan).dropna()
    rho = (
        float(spearmanr(subset[component], subset[outcome]).statistic)
        if len(subset) >= 3 and subset[component].nunique() > 1 and subset[outcome].nunique() > 1 else np.nan
    )
    replication_rows.append({
        "modality": "chest_radiography", "outcome": outcome, "component": component,
        "expected_sign": "positive" if expected_sign > 0 else "negative", "n_edges": len(subset),
        "spearman_rho": rho, "sign_replicated": bool(np.isfinite(rho) and np.sign(rho) == expected_sign),
        "status": "DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION",
    })
replication_table = pd.DataFrame(replication_rows)
write_csv(DISCOVERY_ROOT / "Stage8B_Stage7_Candidate_Sign_Replication_Chest_v0.1.csv", replication_table)

parent_library = pd.read_csv(STAGE8_EDGE_LIBRARY_PATH)
parent_library["origin_stage"] = parent_library.get("origin_stage", "Stage7_or_Stage8")
new_edges = chest_edge_matrix.copy()
new_edges["origin_stage"] = "Stage8B"
common_columns = sorted(set(parent_library.columns) | set(new_edges.columns))
three_modality_library = pd.concat([
    parent_library.reindex(columns=common_columns), new_edges.reindex(columns=common_columns),
], ignore_index=True)
write_csv(DISCOVERY_ROOT / "Stage8B_ThreeModality_Eligible_Edge_Library_v0.1.csv", three_modality_library)
actual_modalities = sorted(three_modality_library["modality"].dropna().unique().tolist())
print("Evaluated chest edges:", len(chest_edge_matrix))
print("Expanded eligible edge library:", len(three_modality_library))
print("Actual modalities represented:", actual_modalities)
display(chest_edge_matrix[[
    "source", "target", "target_auc", "target_auc_ci_lower", "target_ece10",
    "target_balanced_accuracy_at_0_5", "support_fraction", "domain_auc", "observed_transportability_state",
]] if len(chest_edge_matrix) else chest_edge_matrix)
display(replication_table)


Evaluated chest edges: 6
Expanded eligible edge library: 21
Actual modalities represented: ['chest_radiography', 'dermoscopy', 'retinal_fundus']


,source,target,target_auc,target_auc_ci_lower,target_ece10,target_balanced_accuracy_at_0_5,support_fraction,domain_auc,observed_transportability_state
0,TBX11K,Montgomery_CXR,0.561265,0.413937,0.375075,0.530962,0.392857,1.000000,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
1,TBX11K,Shenzhen_CXR,0.643969,0.560362,0.247119,0.598574,0.933735,0.981873,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
2,Montgomery_CXR,TBX11K,0.629656,0.548548,0.640584,0.555070,0.108000,0.999851,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
3,Montgomery_CXR,Shenzhen_CXR,0.727484,0.646340,0.392213,0.561472,0.000000,1.000000,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED
4,Shenzhen_CXR,TBX11K,0.771875,0.717813,0.332839,0.689822,0.646000,0.987854,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED
5,Shenzhen_CXR,Montgomery_CXR,0.797101,0.645576,0.240098,0.664032,0.000000,1.000000,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED


,modality,outcome,component,expected_sign,n_edges,spearman_rho,sign_replicated,status
0,chest_radiography,target_auc,target_mean_knn_distance,negative,6,0.257143,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
1,chest_radiography,source_minus_target_auc,atc_estimated_accuracy,positive,6,0.771429,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
2,chest_radiography,target_ece10,unlabeled_mixture_prevalence,negative,6,0.428571,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
3,chest_radiography,calibration_ece_degradation,atc_estimated_accuracy,positive,6,0.485714,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
4,chest_radiography,target_balanced_accuracy_at_0_5,unlabeled_mixture_prevalence,positive,6,0.085714,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION


In [7]:
#@title 08B-6. Freeze the completion decision, integrity record, report, and storage audit
import matplotlib.pyplot as plt


ready_nlm = sorted(set(acquisition_status.loc[acquisition_status["status"].eq("READY"), "dataset"]))
recoverable_nlm = sorted(set(source_recoverability.loc[
    source_recoverability["source"].isin(["Montgomery_CXR", "Shenzhen_CXR"]) &
    source_recoverability["recoverable"].fillna(False), "source"
]))
access_gate = ready_nlm == ["Montgomery_CXR", "Shenzhen_CXR"]
source_gate = len(recoverable_nlm) >= 1
edge_gate = len(chest_edge_matrix) >= 4
modality_gate = set(actual_modalities) == {"retinal_fundus", "dermoscopy", "chest_radiography"}
if access_gate and source_gate and edge_gate and modality_gate:
    decision = "CHEST_ACCESS_COMPLETED_THREE_MODALITY_EDGE_LIBRARY_ESTABLISHED_READY_TO_FREEZE_HIERARCHICAL_DDO2_SPEC"
elif access_gate and len(chest_edge_matrix) >= 2 and modality_gate:
    decision = "CHEST_ACCESS_COMPLETED_EXTERNAL_EDGES_ADDED_BUT_NLM_SOURCE_GATE_INCOMPLETE"
else:
    decision = "NLM_ACCESS_OR_CHEST_EDGE_COMPLETION_REMAINS_INCOMPLETE"

figure_path = RESULT_ROOT / "Stage8B_Chest_Transportability_Completion_v0.1.png"
if len(chest_edge_matrix) and not figure_path.is_file():
    fig, axes_plot = plt.subplots(1, 2, figsize=(11, 4.5))
    for source, subset in chest_edge_matrix.groupby("source"):
        axes_plot[0].scatter(subset["target_mean_knn_distance_normalised"], subset["target_auc"], s=75, alpha=0.85, label=source)
        axes_plot[1].scatter(subset["unlabeled_mixture_prevalence"], subset["target_ece10"], s=75, alpha=0.85, label=source)
    axes_plot[0].set(xlabel="Normalised target mean kNN distance", ylabel="Target AUC")
    axes_plot[1].set(xlabel="Unlabelled mixture prevalence", ylabel="Target ECE10")
    for axis in axes_plot:
        axis.grid(alpha=0.2); axis.legend(fontsize=8)
    fig.suptitle("Stage 8B chest-radiography transportability completion")
    fig.tight_layout(); fig.savefig(figure_path, dpi=180, bbox_inches="tight"); plt.close(fig)

report_lines = [
    "# Stage 8B — NLM Chest-Radiography Access Completion", "",
    f"- Decision: `{decision}`",
    f"- Parent Stage 8 final hash: `{stage8_claim}`",
    f"- NLM datasets ready: `{ready_nlm}`",
    f"- Recoverable NLM sources: `{recoverable_nlm}`",
    f"- Evaluated chest edges: `{len(chest_edge_matrix)}`",
    f"- Expanded eligible edges: `{len(three_modality_library)}`",
    f"- Actual modalities: `{actual_modalities}`", "",
    "## Integrity boundary", "",
    "The completed Stage 8 record was verified and not modified. Stage 8B used only the explicit official NLM index.html endpoints to repair the two recorded 403 directory-root access failures.", "",
    "## Modelling boundary", "",
    "TBX11K embeddings and source axis were reused without refitting. New NLM source axes used the unchanged Stage 8 representation, linear-probe specification, grouped split, and recoverability gate. All chest target predictions were frozen before outcome evaluation.", "",
    "## Next gate", "",
    "A hierarchical DDO2 specification may be frozen only if the final decision establishes a genuine three-modality edge library. This stage does not fit DDO2.",
]
report_path = RESULT_ROOT / "Stage8B_NLM_Access_Completion_Report_v0.1.md"
report_text = "\n".join(report_lines)
if report_path.is_file():
    assert report_path.read_text(encoding="utf-8") == report_text
else:
    report_path.write_text(report_text, encoding="utf-8")

output_candidates = sorted([
    path for path in STAGE8B_ROOT.rglob("*") if path.is_file()
    and path not in {RUNTIME_STATE_PATH, FINAL_RECORD_PATH}
    and "Output_Integrity_Manifest" not in path.name
], key=str)
output_manifest = pd.DataFrame([{
    "relative_path": str(path.relative_to(STAGE8B_ROOT)), "size_bytes": int(path.stat().st_size),
    "sha256": sha256_file(path),
} for path in output_candidates])
output_manifest_path = RESULT_ROOT / "Stage8B_Output_Integrity_Manifest_v0.1.csv"
write_csv(output_manifest_path, output_manifest)
new_bytes = int(sum(path.stat().st_size for path in output_candidates) + output_manifest_path.stat().st_size)
assert new_bytes <= MAXIMUM_NEW_STAGE8B_BYTES, f"Stage 8B exceeded storage cap: {new_bytes}"

final_payload = {
    "stage": "Stage8B", "decision": decision,
    "scope": "APPEND_ONLY_NLM_ACCESS_COMPLETION_AND_CHEST_EDGE_EXPANSION",
    "parent_stage8_final_record_sha256": stage8_claim,
    "parent_stage8_modified": False,
    "protocol_seal_sha256": seal_payload["seal_sha256"],
    "prediction_freeze_sha256": freeze_payload["freeze_sha256"],
    "ready_nlm_datasets": ready_nlm, "recoverable_nlm_sources": recoverable_nlm,
    "recoverable_chest_sources": recoverable_sources,
    "chest_edges": int(len(chest_edge_matrix)),
    "expanded_eligible_edges": int(len(three_modality_library)),
    "actual_modalities": actual_modalities,
    "chest_state_counts": chest_edge_matrix["observed_transportability_state"].value_counts().astype(int).to_dict() if len(chest_edge_matrix) else {},
    "stage7_candidate_sign_replication_chest": json.loads(replication_table.to_json(orient="records")),
    "nlm_images_accessed": bool(len(embedding_paths)), "images_copied_to_drive": False,
    "target_performance_observed_before_freeze": False,
    "target_model_refit": False, "threshold_tuned": False, "final_ddo2_fitted": False,
    "new_stage8b_bytes": new_bytes, "maximum_new_stage8b_bytes": MAXIMUM_NEW_STAGE8B_BYTES,
    "output_integrity_manifest_sha256": sha256_file(output_manifest_path),
    "next_step": (
        "FREEZE_HIERARCHICAL_MODALITY_CONDITIONAL_DDO2_SPECIFICATION"
        if decision.startswith("CHEST_ACCESS_COMPLETED_THREE_MODALITY")
        else "REASSESS_CHEST_SOURCE_RECOVERABILITY_OR_ACCESS_WITHOUT_TARGET_TUNING"
    ),
    "completed_utc": seal_payload["sealed_utc"],
}
final_payload["final_record_sha256"] = sha256_json(final_payload)
if FINAL_RECORD_PATH.is_file():
    with FINAL_RECORD_PATH.open("r", encoding="utf-8") as handle:
        assert json.load(handle) == final_payload
else:
    atomic_json(FINAL_RECORD_PATH, final_payload)

runtime_state.update({
    "stage8b_complete": True, "decision": decision, "parent_stage8_modified": False,
    "nlm_images_accessed": bool(len(embedding_paths)), "images_copied_to_drive": False,
    "target_performance_observed_before_freeze": False, "target_model_refit": False,
    "threshold_tuned": False, "final_ddo2_fitted": False, "new_stage8b_bytes": new_bytes,
    "final_record_sha256": final_payload["final_record_sha256"], "last_updated_utc": utc_now(),
})
atomic_json(RUNTIME_STATE_PATH, runtime_state)

if TEMP_ROOT.is_dir():
    shutil.rmtree(TEMP_ROOT)

print("\n================ STAGE 8B NLM ACCESS COMPLETION COMPLETE ================")
display(source_recoverability)
display(chest_edge_matrix[[
    "source", "target", "target_auc", "target_auc_ci_lower", "target_ece10",
    "target_balanced_accuracy_at_0_5", "support_fraction", "domain_auc", "observed_transportability_state",
]] if len(chest_edge_matrix) else chest_edge_matrix)
display(replication_table)
print("Decision:", decision)
print("Final record:", FINAL_RECORD_PATH)
print("Final record hash:", final_payload["final_record_sha256"])
print("New Stage 8B Drive storage (MiB):", new_bytes / 1024**2)
print("Parent Stage 8 modified: False")
print("Original NLM images copied to Drive: False")
print("Target refit / threshold tuning / final DDO2 fit: False / False / False")
print("\nSTOP. Interpret Stage 8B before freezing a hierarchical DDO2 specification.")



================ STAGE 8B NLM ACCESS COMPLETION COMPLETE ================


,source,development_units,validation_units,development_oof_auc,development_oof_auc_ci_lower,development_oof_auc_ci_upper,validation_auc,validation_auc_ci_lower,validation_auc_ci_upper,recoverable,failure_reason,axis_provenance,axis_path,axis_sha256,maximum_axis_equivalence_error
0,TBX11K,1500,500,0.958827,0.940773,0.975102,0.963788,0.939730,0.983316,True,,REUSED_SEALED_STAGE8_WITHOUT_REFIT,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,6fc63f746731f607173d7aff4ffdfb2e580be2a4abe5b6...,1.643130e-14
1,Montgomery_CXR,82,56,0.777508,0.671599,0.878600,0.835310,0.721985,0.930683,True,,FIT_ON_STAGE8B_NLM_DEVELOPMENT_ONLY,03_Frozen_Source_Axes/Montgomery_CXR_Frozen_So...,aed9f440093253886d85a11134bba078c195ca6f688f85...,8.881784e-15
2,Shenzhen_CXR,496,166,0.871595,0.837057,0.899187,0.870944,0.812921,0.920921,True,,FIT_ON_STAGE8B_NLM_DEVELOPMENT_ONLY,03_Frozen_Source_Axes/Shenzhen_CXR_Frozen_Sour...,11ebbbc4d42c5fe29d3ade72e0350ef681623862c6f3e1...,1.243450e-14


,source,target,target_auc,target_auc_ci_lower,target_ece10,target_balanced_accuracy_at_0_5,support_fraction,domain_auc,observed_transportability_state
0,TBX11K,Montgomery_CXR,0.561265,0.413937,0.375075,0.530962,0.392857,1.000000,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
1,TBX11K,Shenzhen_CXR,0.643969,0.560362,0.247119,0.598574,0.933735,0.981873,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
2,Montgomery_CXR,TBX11K,0.629656,0.548548,0.640584,0.555070,0.108000,0.999851,DISCRIMINATION_FAILURE_WITH_OR_WITHOUT_ADDITIO...
3,Montgomery_CXR,Shenzhen_CXR,0.727484,0.646340,0.392213,0.561472,0.000000,1.000000,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED
4,Shenzhen_CXR,TBX11K,0.771875,0.717813,0.332839,0.689822,0.646000,0.987854,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED
5,Shenzhen_CXR,Montgomery_CXR,0.797101,0.645576,0.240098,0.664032,0.000000,1.000000,DISCRIMINATION_RETAINED_BUT_CALIBRATION_FAILED


,modality,outcome,component,expected_sign,n_edges,spearman_rho,sign_replicated,status
0,chest_radiography,target_auc,target_mean_knn_distance,negative,6,0.257143,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
1,chest_radiography,source_minus_target_auc,atc_estimated_accuracy,positive,6,0.771429,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
2,chest_radiography,target_ece10,unlabeled_mixture_prevalence,negative,6,0.428571,False,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
3,chest_radiography,calibration_ece_degradation,atc_estimated_accuracy,positive,6,0.485714,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION
4,chest_radiography,target_balanced_accuracy_at_0_5,unlabeled_mixture_prevalence,positive,6,0.085714,True,DESCRIPTIVE_REPLICATION_NOT_FINAL_VALIDATION


Decision: CHEST_ACCESS_COMPLETED_THREE_MODALITY_EDGE_LIBRARY_ESTABLISHED_READY_TO_FREEZE_HIERARCHICAL_DDO2_SPEC
Final record: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Cross_Modal/Stage8B_NLM_Chest_Radiography_Access_Completion_v0.1/06_Results/Stage8B_NLM_Access_Completion_Complete_v0.1.json
Final record hash: 4339402d843af177ef1181df1e3ae4fb7b0b6bd0211e250bb22cd9b255141024
New Stage 8B Drive storage (MiB): 6.557819366455078
Parent Stage 8 modified: False
Original NLM images copied to Drive: False
Target refit / threshold tuning / final DDO2 fit: False / False / False

STOP. Interpret Stage 8B before freezing a hierarchical DDO2 specification.
